In [1]:
import numpy as np
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,classification_report

In [2]:
df = pd.read_csv('data/complaints.csv')
df.head()

,ticket_id,user_id,timestamp,description,location,wing,priority,status,true_wing
0,TICK_1061,STU00016,2026-04-05 18:32,I am facing an issue where the network is not ...,Residencies (A) : Room No (129-B),IT,Low,Pending,IT
1,TICK_1964,STU00004,2026-04-13 03:39,water leaking continuously from pipe,Residencies (B) : Room No (350-A),Plumbing,Medium,Pending,Plumbing
2,TICK_1882,STU00012,2026-04-13 14:31,Currently experiencing an issue with the socke...,Residencies (B) : Room No (393-A),Electrical,Medium,Pending,Electrical
3,TICK_1749,FAC001,2026-03-27 13:05,There is something wrong with the cupboard is ...,Academic Block (6) : Room No (XYZ-153),Carpentry,Medium,In Progress,Carpentry
4,TICK_1599,STU00004,2026-03-26 22:31,There seems to be a problem with the switchboa...,Residencies (A) : Room No (394-B),Electrical,High,In Progress,Electrical


In [3]:
X = df['description']
y = df['true_wing']
print(X.shape,y.shape)

(1000,) (1000,)


In [4]:
vectorizer = TfidfVectorizer(stop_words='english',
    ngram_range=(1,2),   # VERY IMPORTANT
    max_df=0.9,
    min_df=2)
X_vec = vectorizer.fit_transform(X)

In [9]:
X_train,X_test,y_train,y_test = train_test_split(X_vec,y,test_size = 0.2,random_state = 42 , stratify = y)

In [10]:
model = MultinomialNB()
model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None


In [11]:
y_pred = model.predict(X_test)

In [12]:
print("Accuracy:",accuracy_score(y_test,y_pred))
print(classification_report(y_test,y_pred))

Accuracy: 0.865
              precision    recall  f1-score   support

   Carpentry       1.00      0.76      0.86        25
    Cleaning       1.00      0.82      0.90        33
  Electrical       0.82      0.93      0.88        45
        HVAC       1.00      0.85      0.92        26
          IT       1.00      0.84      0.91        31
    Plumbing       0.67      0.93      0.78        40

    accuracy                           0.86       200
   macro avg       0.92      0.85      0.87       200
weighted avg       0.89      0.86      0.87       200



## Testing the data for real time input

In [13]:
def predict_issue(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

print(predict_issue("water leaking in bathroom"))

Plumbing


### Preparing the model for deployment

In [16]:
import pickle
# Save model
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)

# Save vectorizer
with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)